# **Cuarto conjunto de tareas a realizar**

## Paquetes necesarios e inicializaciones

La siguiente práctica consta dos partes principales, la primera de ellas basada en YOLO y en detección de matrículas personas y vehículos y la segunda en OCR (Optical Character Recognition).

Si se tiene una tarjeta gráfica de NVIDIA se puede utilizar la GPU haciendo uso de CUDA, para instalar CUDAv11.6 hacer uso del siguiente script.

In [2]:
import cv2
import math
import yaml
from collections import defaultdict
import numpy as np
from ultralytics import YOLO
import pandas as pd
import os
from pathlib import Path
import shutil
import kagglehub

c:\Users\ivanp\anaconda3\envs\FSI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ================= CONFIGURACIÓN =================
KAGGLE_DATASET_ID = 'pkdarabi/cardetection'
# Ruta donde quieres guardar el dataset FINAL y LIMPIO
TARGET_DIR = r"C:/Users/ivanp/Desktop/traffic_signals_without_10/TGC_RBNW"

# Configuración de limpieza
CLASS_ID_TO_DELETE = 2  # La clase "Speed Limit 10"

def copy_dataset(src, dst):
    """Copia el dataset descargado a la carpeta de destino."""
    print(f"\n--- Copiando archivos a: {dst} ---")
    
    # Si la carpeta ya existe, se borra para asegurar un inicio limpio
    if os.path.exists(dst):
        print(f"La carpeta destino ya existe. Borrando para empezar de cero...")
        shutil.rmtree(dst)
    
    # Copiar todo el contenido a partir de la capreta 'car' si existe
    src_car_path = os.path.join(src, "car")
    
    if os.path.exists(src_car_path):
        shutil.copytree(src_car_path, dst)
    else:
        # Si no tiene subcarpeta 'car', copiar todo directamente
        shutil.copytree(src, dst)
        
    print("Copia finalizada.")

def clean_labels(base_dir):
    """
    Recorre los archivos, borra la clase 2, reordena IDs y elimina vacíos.
    """
    print(f"\n--- Iniciando LIMPIEZA de clase {CLASS_ID_TO_DELETE} en {base_dir} ---")
    
    stats = {'deleted_files': 0, 'modified_files': 0}
    splits = ['train', 'valid', 'test'] # Kaggle suele usar 'valid' o 'val'

    for split in splits:
        labels_path = Path(base_dir) / split / 'labels'
        images_path = Path(base_dir) / split / 'images'
        
        if not labels_path.exists():
            continue # Saltar si no existe

        print(f"Procesando: {split}...")
        
        txt_files = list(labels_path.glob('*.txt'))
        
        for txt_file in txt_files:
            with open(txt_file, 'r') as f:
                lines = f.readlines()

            new_lines = []
            modified = False
            has_content = False # Para saber si queda alguna etiqueta válida

            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                class_id = int(parts[0])
                coords = parts[1:]

                if class_id == CLASS_ID_TO_DELETE:
                    modified = True # Se borra esta línea
                
                elif class_id > CLASS_ID_TO_DELETE:
                    # Reordenar: ID - 1
                    new_id = class_id - 1
                    new_lines.append(f"{new_id} {' '.join(coords)}\n")
                    modified = True
                    has_content = True
                
                else:
                    # ID menor: Se mantiene igual
                    new_lines.append(line)
                    has_content = True

            # --- ACCIONES ---
            if not has_content and len(lines) > 0:
                # Si tenía etiquetas y ahora no tiene ninguna -> Borrar archivo e imagen
                txt_file.unlink()
                
                # Buscar y borrar la imagen asociada (.jpg, .png, etc)
                img_name = txt_file.stem
                for ext in ['.jpg', '.jpeg', '.png']:
                    img_candidate = images_path / (img_name + ext)
                    if img_candidate.exists():
                        img_candidate.unlink()
                        break
                
                stats['deleted_files'] += 1
            
            elif modified:
                # Si hubo cambios y aún hay contenido -> Sobrescribir
                with open(txt_file, 'w') as f:
                    f.writelines(new_lines)
                stats['modified_files'] += 1

    print(f"Limpieza terminada. Modificados: {stats['modified_files']}, Eliminados (vacíos): {stats['deleted_files']}")

def generate_csv(base_dir):
    """Genera el CSV final con los datos que quedaron."""
    print("\n--- Generando CSV final ---")
    datos = []
    
    for split in ['train', 'valid', 'test']:
        labels_dir = os.path.join(base_dir, split, 'labels')
        
        if not os.path.exists(labels_dir): continue
        
        for txt_file in os.listdir(labels_dir):
            if not txt_file.endswith('.txt'): continue
            
            full_path = os.path.join(labels_dir, txt_file)
            # Asumimos jpg por defecto para el nombre en el CSV
            img_name = txt_file.replace('.txt', '.jpg')
            
            try:
                with open(full_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            datos.append({
                                'nombre_archivo': img_name,
                                'clase_indice': int(parts[0]),
                                'x_center': float(parts[1]),
                                'y_center': float(parts[2]),
                                'width': float(parts[3]),
                                'height': float(parts[4]),
                                'subset': split
                            })
            except Exception as e:
                pass

    df = pd.DataFrame(datos)
    csv_path = os.path.join(base_dir, 'dataset_limpio.csv')
    df.to_csv(csv_path, index=False)
    
    print(f"Total de etiquetas válidas: {len(df)}")
    if not df.empty:
        print("Conteo por clase:")
        print(df['clase_indice'].value_counts().sort_index())
    print(f"CSV guardado en: {csv_path}")

# --- EJECUCIÓN PRINCIPAL ---
if __name__ == "__main__":
    # 1. Descargar
    print(f"Descargando dataset: {KAGGLE_DATASET_ID}")
    download_path = kagglehub.dataset_download(KAGGLE_DATASET_ID)
    
    # 2. Copiar a tu escritorio
    copy_dataset(download_path, TARGET_DIR)
    
    # 3. Limpiar (Borrar clase 2, reordenar, borrar vacíos)
    clean_labels(TARGET_DIR)
    
    # 4. Generar CSV de comprobación
    generate_csv(TARGET_DIR)
    
    print(f"\n¡LISTO! Tu dataset limpio está en: {TARGET_DIR}")

Descargando dataset: pkdarabi/cardetection

--- Copiando archivos a: C:/Users/ivanp/Desktop/traffic_signals_without_10/TGC_RBNW ---
La carpeta destino ya existe. Borrando para empezar de cero...
Copia finalizada.

--- Iniciando LIMPIEZA de clase 2 en C:/Users/ivanp/Desktop/traffic_signals_without_10/TGC_RBNW ---
Procesando: train...
Procesando: valid...
Procesando: test...
Limpieza terminada. Modificados: 4168, Eliminados (vacíos): 10

--- Generando CSV final ---
Total de etiquetas válidas: 5990
Conteo por clase:
clase_indice
0     774
1     787
2     365
3     139
4     356
5     387
6     468
7     343
8     404
9     422
10    449
11    440
12    240
13    416
Name: count, dtype: int64
CSV guardado en: C:/Users/ivanp/Desktop/traffic_signals_without_10/TGC_RBNW\dataset_limpio.csv

¡LISTO! Tu dataset limpio está en: C:/Users/ivanp/Desktop/traffic_signals_without_10/TGC_RBNW


In [3]:
# === ENTRENAMIENTO DEL MODELO ===
print("Iniciando entrenamiento del modelo")

model = YOLO("yolo11m.pt")

# Entrenamiento con el dataset de matrículaas personalizado
results = model.train(
    data="data.yaml",               # Ruta al archivo de configuración de datos
    imgsz=640,                      # Tamaño de las imágenes (Se reescalan para más eficiencia)
    epochs=50,                      # Número de épocas de entrenamiento
    project="runs/train_custom",    # Directorio donde se guardarán los resultados
    name="exp1",                    # Nombre del entrenamiento
    exist_ok=True,                  # Sobrescribir si el directorio ya existe
    plots=True                      # Generar gráficos de métricas durante el entrenamiento
)

print("Entrenamiento completado.")

Iniciando entrenamiento del modelo
New https://pypi.org/project/ultralytics/8.3.231 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.230  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp1, n

In [4]:
# === CARGAR CLASES PERSONALIZADAS ===
with open("data.yaml", "r") as f:
    data = yaml.safe_load(f)
classNames = data["names"]

print(f"Clases personalizadas: {classNames}")

# === CARGAR EL MODELO ENTRENADO ===
model = YOLO("runs/train_custom/exp1/weights/best.pt")

# === TRACKING EN VÍDEO ===
print("🎥 Iniciando detección y tracking...")

video_path = "../Resources/test.mp4"
output_path = "../outputs/tracking_result_YOLO.mp4"

vid = cv2.VideoCapture(video_path)

if not vid.isOpened():
    print("Error: no se pudo abrir el video.")
    exit()

# === Crear escritor de video ===
fps = int(vid.get(cv2.CAP_PROP_FPS))
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

track_history = defaultdict(lambda: [])

while True:
    ret, frame = vid.read()
    if not ret:
        print("Fin del video.")
        break

    # Detección con seguimiento
    results = model.track(frame, persist=True, stream=True)

    for r in results:
        boxes = r.boxes
        for box in boxes:
            # Coordenadas del contenedor
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = math.ceil((box.conf[0] * 100)) / 100
            cls = int(box.cls[0])
            class_name = classNames[cls]

            # ID de seguimiento
            track_id = ""
            if hasattr(box, "id") and box.id is not None:
                track_id = str(int(box.id[0].tolist()))

            # Color en función de la clase
            escala = int((cls / len(classNames)) * 255 * 3)
            if escala >= 255 * 2:
                R, G, B = 255, 255, escala - 255 * 2
            elif escala >= 255:
                R, G, B = 255, escala - 255, 0
            else:
                R, G, B = escala, 0, 0

            # Dibujar caja y texto
            cv2.rectangle(frame, (x1, y1), (x2, y2), (R, G, B), 3)
            cv2.putText(
                frame,
                f"{track_id} {class_name} {confidence}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )

            # Guardar historial de trayectorias
            if hasattr(box, "id") and box.id is not None:
                tid = int(box.id[0])
                x_center = int((x1 + x2) / 2)
                y_center = int((y1 + y2) / 2)
                track = track_history[tid]
                track.append((x_center, y_center))
                if len(track) > 30:
                    track.pop(0)

                points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=2)

    # === Guardar frame procesado en el video ===
    out.write(frame)

vid.release()
out.release()
cv2.destroyAllWindows()

print(f"--- Proceso finalizado correctamente. Video guardado en: {output_path} ---")

Clases personalizadas: ['Green Light', 'Red Light', 'Speed Limit 100', 'Speed Limit 110', 'Speed Limit 120', 'Speed Limit 20', 'Speed Limit 30', 'Speed Limit 40', 'Speed Limit 50', 'Speed Limit 60', 'Speed Limit 70', 'Speed Limit 80', 'Speed Limit 90', 'Stop']
🎥 Iniciando detección y tracking...

0: 640x640 1 Stop, 14.9ms
Speed: 11.4ms preprocess, 14.9ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Stop, 14.3ms
Speed: 2.7ms preprocess, 14.3ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Stop, 15.5ms
Speed: 3.0ms preprocess, 15.5ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 70, 14.9ms
Speed: 2.8ms preprocess, 14.9ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Speed Limit 20, 1 Stop, 15.7ms
Speed: 3.0ms preprocess, 15.7ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Stop, 16.2ms
Speed: 2.7ms preprocess, 16